# NBA shot prediction: baseline walkthrough

Run `uv run --frozen nba-shots baseline --output artifacts/day1` first. This notebook reads saved validation artifacts, reloads models, and never trains or evaluates the test partition.


In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
RUN = ROOT / "artifacts/day1"
assert (RUN / "models/run.json").exists(), "Run the baseline command first."
metrics = pd.read_csv(RUN / "models/validation_metrics.csv")
metrics[["model", "n_scored", "brier_score", "log_loss", "roc_auc", "accuracy"]]


## Source and split audit

The source checksum and exact game assignments are persisted. Whole dates stay in one partition. Outcomes are absent from each model’s feature list.


In [ ]:
audit = json.loads((RUN / "prepared/audit.json").read_text())
splits = json.loads((RUN / "prepared/splits.json").read_text())
print("CSV SHA-256:", audit["csv_sha256"])
print("Missing measurements:", audit["feature_missing_values"])
pd.DataFrame(splits["summary"]).T


## Validation probability calibration

The diagonal is ideal calibration. Brier score and log loss measure overall probability quality; the curve helps inspect reliability directly.


In [ ]:
image_path = RUN / "models/validation_kaggle_2014_15_validation_calibration.png"
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(mpimg.imread(image_path))
ax.axis("off")
plt.show()


## Reloaded predictions on the same validation shots

Each estimator receives only its saved feature list. Labels are added to the display after inference for comparison.


In [ ]:
shots = pd.read_parquet(RUN / "prepared/shots.parquet")
sample = shots.loc[shots["split"].eq("validation")].head(5)
comparison = sample[["shot_id", "y_true"]].copy()
for name in ["constant", "logistic_regression", "xgboost"]:
    artifact = joblib.load(RUN / "models" / f"{name}.joblib")
    assert "y_true" not in artifact["features"]
    comparison[name] = artifact["pipeline"].predict_proba(sample[artifact["features"]])[:, 1]
comparison


## Next experiment

Register a separate recent-video cohort, annotate release frames, and use `extract-cv`, `train --cv`, and `predict-llm`. Audit the saved release overlays before reporting a CV improvement. The video and agent commands are documented in `docs/experiments.md`; this baseline does not establish their performance.
